# Spatial Biomarker Analysis

This notebook conducts spatial analysis and biomarker discovery
on the Cellpose-SAM segmented Xenium data. It includes spatial
autocorrelation (Moran's I), differential expression analysis,
and biomarker candidate identification.


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cpsam_xenium_analysis import config
from cpsam_xenium_analysis.data_loader import load_transcripts
from cpsam_xenium_analysis.analysis import SpatialAnalyzer, BiomarkerDiscovery, CellTyper
from cpsam_xenium_analysis.visualization import plots as vis
from cpsam_xenium_analysis.segmentation.post_process import compute_cell_centroids
from scipy.sparse import load_npz

%matplotlib inline


In [ ]:
# Load CPSAM morphology results
gene_names = np.load(config.OUTPUT_DIR / 'expr_cpsam_morphology_gene_names.npy', allow_pickle=True).tolist()
cell_labels = np.load(config.OUTPUT_DIR / 'expr_cpsam_morphology_cell_ids.npy')
expr_mat = load_npz(str(config.OUTPUT_DIR / 'expr_cpsam_morphology_matrix.npz'))
mask_morph = np.load(config.OUTPUT_DIR / 'masks_cpsam_morphology.npy')
transcripts = load_transcripts(min_qv=20)

print(f'Expression matrix: {expr_mat.shape}')
print(f'Cells: {len(cell_labels)}')
print(f'Genes: {len(gene_names)}')


## 1. Spatial Autocorrelation (Moran's I)


In [ ]:
# Get cell centroids
centroids = compute_cell_centroids(mask_morph)
coords = np.array([[centroids[lbl][1], centroids[lbl][0]] 
    for lbl in cell_labels if lbl in centroids])
print(f'Coordinates: {coords.shape}')


In [ ]:
spatial = SpatialAnalyzer(k_neighbors=15)
spatial.compute_spatial_weights(coords)

# Compute Moran's I for all genes
morans_df = spatial.compute_all_morans_i(expr_mat, gene_names)
print(f'Genes with significant spatial autocorrelation: {(morans_df["p_value"] < 0.05).sum()}')
morans_df.head(10)


In [ ]:
# Visualize Moran's I
fig = vis.plot_morans_i_scatter(morans_df, 
    highlight_genes=['INS', 'GCG', 'KRT19', 'EPCAM'],
    title="Moran's I - Spatial Autocorrelation of Gene Expression"
)


## 2. Spatially Variable Genes


In [ ]:
svg_df = spatial.spatially_variable_genes(expr_mat, gene_names)
print(f'Top 15 spatially variable genes:')
svg_df.head(15)


In [ ]:
fig = vis.plot_spatial_variable_genes(svg_df, n_top=20, 
    title='Top 20 Spatially Variable Genes (by Moran\'s I)'
)


## 3. Spatial Expression of Key Markers


In [ ]:
crop = config.CROP_ROI
aligner = __import__('cpsam_xenium_analysis.integration.alignment', fromlist=['CoordinateAligner']).CoordinateAligner()
transcripts_px = aligner.transcripts_to_pixel_coords(transcripts, crop_roi=crop)

markers = ['INS', 'GCG', 'KRT19', 'PTPRC', 'PECAM1', 'SST']
markers = [g for g in markers if g in gene_names]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, gene in enumerate(markers):
    gene_tr = transcripts_px[transcripts_px['feature_name'] == gene]
    axes[i].imshow((mask_morph > 0).astype(np.uint8), cmap='gray', alpha=0.3)
    if len(gene_tr) > 0:
        axes[i].scatter(gene_tr['x_pixel'], gene_tr['y_pixel'], 
            s=0.5, c='red', alpha=0.5)
    axes[i].set_title(f'{gene} ({len(gene_tr)} transcripts)')
    axes[i].axis('off')
    axes[i].invert_yaxis()

plt.suptitle('Spatial Expression of Key Pancreatic Markers', fontsize=14)
plt.tight_layout()
plt.show()


## 4. Differential Expression Analysis


In [ ]:
# First run cell typing to get labels
typer = CellTyper()
cell_types = typer.score_cell_types(expr_mat, gene_names)

# Create binary labels for DE
type_labels = cell_types['cell_type'].values
binary_labels = np.array([
    'Tumor' if t == 'Tumor_Epithelial' else 'Immune' if t == 'Immune' else 'Other'
    for t in type_labels
])


In [ ]:
bio = BiomarkerDiscovery()

de_results = bio.differential_expression(
    expr_mat, gene_names, binary_labels, 
    group_a='Tumor', group_b='Other',
)
print(f'Significant genes: {(de_results["p_value_adj"] < 0.05).sum()}')
de_results.head(10)


In [ ]:
# Volcano plot
fig = vis.plot_biomarker_volcano(de_results,
    title='Differential Expression: Tumor Epithelial vs Other Cells'
)


## 5. Gene-Morphology Correlation


In [ ]:
from cpsam_xenium_analysis.segmentation.post_process import compute_morphology_features

morph_features = compute_morphology_features(mask_morph)

corr_df = bio.correlate_with_morphology(
    expr_mat, gene_names, morph_features, cell_labels, feature_name='area'
)
print('\nTop genes correlated with cell area:')
corr_df.head(10)


## 6. Surrogate Biomarker Candidates


In [ ]:
candidates = bio.surrogate_biomarkers(
    expr_mat, gene_names, morans_df, de_results
)
print('\nTop biomarker candidates (combining DE + spatial signals):')
candidates[['gene', 'log2fc', 'p_value_adj', 'morans_i', 'biomarker_score']].head(20)


## 7. Gene Signature Scoring


In [ ]:
signatures = {
    'Epithelial': ['EPCAM', 'KRT19', 'KRT7', 'CDH1'],
    'Mesenchymal': ['VIM', 'FN1', 'COL1A1', 'SNAI2'],
    'Immune_activation': ['CD3D', 'CD8A', 'GZMB', 'IFNG'],
    'Angiogenesis': ['VEGFA', 'PECAM1', 'CDH5', 'VWF'],
    'Proliferation': ['MKI67', 'TOP2A', 'PCNA'],
}
# Filter to available genes
signatures = {k: [g for g in v if g in gene_names] for k, v in signatures.items()}
signatures = {k: v for k, v in signatures.items() if len(v) > 0}
print('Available signatures:', list(signatures.keys()))


In [ ]:
sig_scores = bio.gene_signature_scoring(expr_mat, gene_names, signatures)
sig_scores.head()


In [ ]:
# Map signature scores back to spatial coordinates
valid_idx = [i for i, lbl in enumerate(cell_labels) if lbl in centroids]
valid_coords = np.array([centroids[cell_labels[i]] for i in valid_idx])

fig, axes = plt.subplots(1, len(signatures), figsize=(4*len(signatures), 4))
if len(signatures) == 1:
    axes = [axes]

for i, sig in enumerate(signatures.keys()):
    scores = sig_scores[f'{sig}_zscore'].values[valid_idx]
    sc = axes[i].scatter(valid_coords[:, 1], valid_coords[:, 0], 
        c=scores, cmap='RdBu_r', s=3, vmin=-2, vmax=2)
    axes[i].set_title(sig)
    axes[i].axis('off')
    axes[i].invert_yaxis()
    plt.colorbar(sc, ax=axes[i], shrink=0.6)

plt.suptitle('Spatial Distribution of Gene Signatures', fontsize=14)
plt.tight_layout()
plt.show()
